# NeoProphet starter

Fit a model on the Peyton Manning Wikipedia page views dataset, forecast a year ahead, inspect the components and check accuracy with cross-validation.

Run with the **NeoProphet (uv)** kernel, which uses the project `.venv` and the editable `prophet` package from `python/`.

In [ ]:
import logging
from pathlib import Path

import pandas as pd

from prophet import Prophet, forecast
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

# cmdstanpy resets its level to DEBUG on first use unless a handler already exists
cmdstan_logger = logging.getLogger("cmdstanpy")
cmdstan_logger.addHandler(logging.StreamHandler())
cmdstan_logger.setLevel(logging.WARNING)

## Load data

Prophet expects two columns: `ds` (timestamp) and `y` (value).

In [ ]:
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples").is_dir())
df = pd.read_csv(ROOT / "examples" / "example_wp_log_peyton_manning.csv", parse_dates=["ds"])
df.head()

## One call

`forecast()` works out the columns and the frequency, fits the model and predicts the horizon. Add `tune=True` to choose the hyperparameters by cross-validation first.

In [ ]:
result = forecast(df, horizon="12 months", country_holidays="US")
print(result.summary())
fig = result.plot()

## Fit and forecast with the full API

The `Prophet` class gives full control; the result is the same model as above without holidays.

In [ ]:
m = Prophet()
m.fit(df)

future = m.make_future_dataframe(periods=365)
forecast = m.predict(future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

In [ ]:
fig = m.plot(forecast)

In [ ]:
fig = m.plot_components(forecast)

## Cross-validation

Train on the first two years, then refit every 180 days and score a one-year horizon.

In [ ]:
df_cv = cross_validation(
    m, initial="730 days", period="180 days", horizon="365 days", disable_tqdm=True
)
df_p = performance_metrics(df_cv)
df_p.head()

In [ ]:
fig = plot_cross_validation_metric(df_cv, metric="mape")